# Notebook 1 — Descarga de Líneas Espectrales
## Fuente: NIST Atomic Spectra Database (ASD)
## Proyecto: Óptica y Fotónica — Primer Parcial

**Estudiante:** Perez Criollo Andres David  
**Fuente oficial:** https://physics.nist.gov/PhysRefData/ASD/lines_form.html  
**Endpoint API:** http://physics.nist.gov/cgi-bin/ASD/lines1.pl  
**Fecha de descarga:** (completar al ejecutar)  

---

### ¿Qué se descarga en este notebook?

Las **líneas espectrales** de los primeros 10 elementos de la tabla periódica  
(H, He, Li, Be, B, C, N, O, F, Ne), en su estado neutro (I) y primer ionizado (II).

Cada fila del resultado representa una **transición electrónica** y contiene:
- Longitud de onda observada y Ritz (en nm)
- Intensidad relativa de la línea
- Coeficiente de Einstein Aki (probabilidad de emisión espontánea, en s⁻¹)
- Fuerza de oscilador log(gf)
- Energía del nivel inferior y superior (en cm⁻¹)
- Configuración electrónica y término espectroscópico de cada nivel
- Tipo de transición (E1 permitida / M1,E2 prohibida)
- Incertidumbres y referencias bibliográficas

---

### Instrucciones
Ejecuta las celdas **en orden de arriba hacia abajo**.  
Al finalizar, el archivo `nist_lines_H_Ne_original.csv` quedará guardado  
en la carpeta `datos_originales/`. **No modificar ese archivo.**

---
## Celda 1 — Instalación de librerías

In [ ]:
# Instalacion de librerias necesarias
# pandas  -> manejo y analisis de datos tabulares
# requests -> hacer peticiones HTTP a la API del NIST

!pip install pandas requests

---
## Celda 2 — Importación de librerías

In [ ]:
import requests          # Para hacer la peticion HTTP a la API del NIST
import pandas as pd      # Para leer el CSV y manipular los datos
from io import StringIO  # Para convertir el texto de la respuesta en un objeto legible por pandas
import os               # Para crear carpetas en el sistema de archivos
from datetime import datetime  # Para registrar la fecha de descarga

print("Librerias importadas correctamente.")
print(f"Fecha y hora de ejecucion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Celda 3 — Crear carpeta de datos originales

In [ ]:
# Crear la carpeta donde se guardaran los archivos originales
# Esta carpeta NO debe ser modificada despues de la descarga

os.makedirs("datos_originales", exist_ok=True)

print("Carpeta 'datos_originales' lista.")

---
## Celda 4 — Definición de la URL y los parámetros de consulta

Cada parámetro corresponde exactamente a una opción del formulario web en:  
https://physics.nist.gov/PhysRefData/ASD/lines_form.html

In [ ]:
# URL del endpoint de la API del NIST ASD para lineas espectrales
URL = "http://physics.nist.gov/cgi-bin/ASD/lines1.pl"

# Elementos a consultar: estado neutro (I) y primer ionizado (II)
# de los primeros 10 elementos de la tabla periodica
# El punto y coma (;) es el separador entre espectros en la API del NIST
ESPECTRA = "H I;He I;Li I;Li II;Be I;Be II;B I;B II;C I;C II;N I;N II;O I;O II;F I;F II;Ne I;Ne II"

parametros = {

    # --- CAMPO PRINCIPAL ---
    "spectra": ESPECTRA,
    # Que elementos y estados de ionizacion consultar
    # Equivale al campo 'Spectrum' en el formulario web

    # --- RANGO DE LONGITUD DE ONDA ---
    "low_w": "",
    # Longitud de onda minima — se deja vacia para obtener TODAS
    # Si se pusiera un valor (ej. 200), solo traeria lineas >= 200 nm

    "upp_w": "",
    # Longitud de onda maxima — se deja vacia para obtener TODAS
    # Equivale al campo 'Upper' en el formulario web

    # --- UNIDADES DE LONGITUD DE ONDA ---
    "unit": 1,
    # 0 = Angstroms (Å)
    # 1 = Nanometros (nm)  <-- SELECCIONADO
    # 2 = Micrometros (µm)
    # Se elige nm porque es la unidad estandar en optica/fotonica
    # y facilita clasificar UV (<380 nm), visible (380-780 nm) e IR (>780 nm)

    # --- FORMATO DE SALIDA ---
    "format": 2,
    # 0 = HTML (solo para ver en navegador, no procesable)
    # 1 = ASCII texto plano
    # 2 = CSV  <-- SELECCIONADO
    # 3 = Tab-delimited
    # CSV es el mas compatible con pandas y MySQL

    # --- QUE LINEAS INCLUIR ---
    "line_out": 0,
    # 0 = Todas las lineas (All)  <-- SELECCIONADO
    # 1 = Solo las que tienen probabilidades de transicion
    # 2 = Solo las que tienen clasificacion de nivel de energia
    # 3 = Solo las que tienen longitudes de onda observadas
    # Se elige 0 para no perder ninguna linea desde el inicio

    # --- UNIDADES DE ENERGIA ---
    "en_unit": 0,
    # 0 = cm-1 (numero de onda inverso)  <-- SELECCIONADO
    # 1 = eV
    # 2 = Rydberg
    # cm-1 es la unidad nativa del NIST para niveles atomicos
    # Se puede convertir a eV en la limpieza si se necesita

    # --- MOSTRAR RESULTADO COMPLETO ---
    "output": 0,
    # 0 = Todo en una sola pagina (in its entirety)  <-- SELECCIONADO
    # 1 = Paginado
    # Se elige 0 para obtener todos los datos de una sola vez

    # --- LONGITUD DE ONDA OBSERVADA ---
    "show_obs_wl": 1,
    # 1 = Incluir longitud de onda observada (medicion experimental real)
    # Es el dato mas valioso: medido directamente en laboratorio

    # --- LONGITUD DE ONDA RITZ ---
    "show_calc_wl": 1,
    # 1 = Incluir longitud de onda Ritz (calculada desde diferencia de niveles)
    # Complementa la observada: hay lineas sin medicion directa
    # pero con Ritz calculada

    # --- INCERTIDUMBRES ---
    "unc_out": 1,
    # 1 = Incluir incertidumbre de la longitud de onda
    # Util para evaluar la calidad/confiabilidad de cada dato

    # --- ORDEN DE SALIDA ---
    "order_out": 0,
    # 0 = Ordenado por longitud de onda  <-- SELECCIONADO
    # 1 = Ordenado por multiplete

    # --- CONVENCION DE LONGITUD DE ONDA (VACIO vs AIRE) ---
    "show_av": 2,
    # 2 = Vacio para < 2000 Å, aire para 2000-10000 Å, numero de onda para > 10000 Å
    # Esta es la convencion estandar del NIST

    # --- COEFICIENTE DE EINSTEIN Aki ---
    "A_out": 0,
    # 0 = Incluir Aki (probabilidad de transicion espontanea en s⁻¹)
    # Este es el parametro mas importante para fotonica:
    # indica que tan facilmente el atomo emite un foton

    # --- INTENSIDADES RELATIVAS ---
    "intens_out": "on",
    # Incluir intensidad relativa de cada linea
    # Indica el brillo relativo de cada linea espectral

    # --- TRANSICIONES PERMITIDAS (E1) ---
    "allowed_out": 1,
    # 1 = Incluir transiciones electricas de dipolo (E1)
    # Son las transiciones mas comunes y brillantes

    # --- TRANSICIONES PROHIBIDAS (M1, E2) ---
    "forbid_out": 1,
    # 1 = Incluir transiciones prohibidas (magneticas y cuadrupolo electrico)
    # Son mas raras pero existen en el dataset y dan info complementaria

    # --- CONFIGURACION ELECTRONICA ---
    "conf_out": "on",
    # Incluir la configuracion electronica de cada nivel
    # Ejemplo: 1s2 2s2 2p1
    # Necesaria para el modelo relacional (tabla Nivel_Energia)

    # --- TERMINO ESPECTROSCOPICO ---
    "term_out": "on",
    # Incluir el termino espectroscopico (ej. 2P*, 3D)
    # Describe el estado cuantico completo del nivel

    # --- ENERGIAS DE LOS NIVELES ---
    "enrg_out": "on",
    # Incluir la energia del nivel inferior y superior de cada transicion
    # Fundamental para calcular la energia del foton emitido

    # --- NUMERO CUANTICO J ---
    "J_out": "on",
    # Incluir el numero cuantico de momento angular total J
    # Necesario para calcular la degeneracion g = 2J+1

    # --- REFERENCIAS BIBLIOGRAFICAS ---
    "bibrefs": 1,
    # 1 = Incluir referencias de las probabilidades de transicion y de las lineas
    # Documenta la fuente cientifica de cada dato — requerido por la rubrica

    # --- PARAMETROS TECNICOS DEL FORMULARIO ---
    "submit": "Retrieve Data",
    "de": 0,
    "plot_out": 0,
    "I_scale_type": 1,
    "page_size": 15,
    "tsb_value": 0,
    "min_str": "",
    "max_str": "",
    "max_low_enrg": "",
    "max_upp_enrg": "",
    "min_accur": "",
    "min_intens": "",
}

print("Parametros de consulta definidos.")
print(f"Elementos a consultar: {ESPECTRA}")
print(f"Total de espectros: {len(ESPECTRA.split(';'))}")

---
## Celda 5 — Ejecutar la petición a la API del NIST

In [ ]:
print("Enviando peticion al NIST ASD...")
print(f"URL: {URL}")
print("Esto puede tardar entre 10 y 60 segundos dependiendo del volumen de datos.")
print("-" * 60)

response = requests.get(URL, params=parametros, timeout=120)

# Verificar que la peticion fue exitosa
print(f"Codigo de respuesta HTTP: {response.status_code}")

if response.status_code == 200:
    print("Peticion exitosa. Datos recibidos.")
    print(f"Tamano de la respuesta: {len(response.text):,} caracteres")
else:
    print(f"ERROR: La peticion fallo con codigo {response.status_code}")
    print(response.text[:500])

---
## Celda 6 — Verificar y previsualizar la respuesta

In [ ]:
# Mostrar las primeras lineas del texto recibido para verificar
# que el formato es CSV correcto

lineas_respuesta = response.text.split("\n")

print(f"Total de lineas en la respuesta: {len(lineas_respuesta):,}")
print("\n--- Primeras 10 lineas de la respuesta ---")
for i, linea in enumerate(lineas_respuesta[:10]):
    print(f"[{i}] {linea}")

---
## Celda 7 — Cargar los datos en un DataFrame de pandas

In [ ]:
# El NIST devuelve el CSV con algunas lineas de encabezado antes de los datos
# Se usa StringIO para que pandas pueda leer el texto directamente
# sin necesidad de guardarlo primero en disco

# Filtrar lineas que son comentarios del NIST (empiezan con # o son vacias)
lineas_datos = []
for linea in lineas_respuesta:
    # El NIST incluye lineas de separacion con guiones — las saltamos
    if linea.strip() and not linea.startswith("---"):
        lineas_datos.append(linea)

texto_limpio = "\n".join(lineas_datos)

# Leer el CSV con pandas
try:
    df_lines = pd.read_csv(
        StringIO(texto_limpio),
        sep=",",
        low_memory=False,    # Evita advertencias con columnas de tipo mixto
        on_bad_lines="warn"  # Avisa si hay filas con problemas sin detener la lectura
    )
    print(f"DataFrame cargado exitosamente.")
    print(f"Filas:    {df_lines.shape[0]:,}")
    print(f"Columnas: {df_lines.shape[1]}")
    print(f"\nNombres de columnas:")
    for col in df_lines.columns:
        print(f"  - {col}")
except Exception as e:
    print(f"Error al parsear el CSV: {e}")
    print("Revisa la celda anterior para ver el formato de la respuesta.")

---
## Celda 8 — Inspección inicial del DataFrame

In [ ]:
# Mostrar las primeras filas del DataFrame
print("=== Primeras 5 filas del dataset ===")
df_lines.head()

In [ ]:
# Tipos de datos de cada columna
print("=== Tipos de datos por columna ===")
print(df_lines.dtypes)

In [ ]:
# Resumen estadistico basico
print("=== Resumen estadistico ===")
df_lines.describe(include="all")

---
## Celda 9 — Guardar el archivo original sin modificaciones

In [ ]:
# Nombre del archivo original
NOMBRE_ARCHIVO = "datos_originales/nist_lines_H_Ne_original.csv"

# Guardar el texto crudo tal como vino del NIST (sin modificar)
# Esto cumple el requisito de conservar el archivo original intacto
with open(NOMBRE_ARCHIVO, "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"Archivo original guardado en: {NOMBRE_ARCHIVO}")
print(f"Tamano del archivo: {os.path.getsize(NOMBRE_ARCHIVO):,} bytes")
print()
print("IMPORTANTE: Este archivo NO debe modificarse.")
print("Es la evidencia de la fuente de datos oficial (NIST ASD).")
print()
print("=" * 60)
print("DOCUMENTACION DE DESCARGA")
print("=" * 60)
print(f"Fuente:          NIST Atomic Spectra Database (ASD)")
print(f"URL base:        {URL}")
print(f"Tipo de dato:    Lineas espectrales")
print(f"Elementos:       H, He, Li, Be, B, C, N, O, F, Ne (neutros e ionizados)")
print(f"Rango lambda:    Completo (sin restriccion)")
print(f"Unidades lambda: Nanometros (nm)")
print(f"Unidades E:      cm-1")
print(f"Formato salida:  CSV")
print(f"Fecha descarga:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Filas obtenidas: {df_lines.shape[0]:,}")
print(f"Columnas:        {df_lines.shape[1]}")
print(f"Archivo:         {NOMBRE_ARCHIVO}")

---
## ✅ Descarga completada

El archivo `datos_originales/nist_lines_H_Ne_original.csv` contiene todas las líneas  
espectrales de los 10 primeros elementos en estado neutro y primer ionizado.

**Siguiente paso:** Ejecutar el Notebook 2 para descargar los niveles de energía.  
**Después:** Notebook de exploración y limpieza con pandas.